# Final Model

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [3]:
data = pd.read_csv(
    "../Data/Processed/solar_model_data.csv"
)

data["time_utc"] = pd.to_datetime(
    data["time_utc"],
    utc=True
)

data["time_sl"] = pd.to_datetime(
    data["time_sl"]
)

data = data.sort_values(
    ["time_utc", "district"]
).reset_index(drop=True)

print("Dataset shape:", data.shape)

Dataset shape: (227218, 21)


Recreate Chronological Split

In [4]:
unique_times = (
    data["time_utc"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

train_end = int(
    len(unique_times) * 0.70
)

validation_end = int(
    len(unique_times) * 0.85
)

train_last_time = unique_times.iloc[
    train_end - 1
]

validation_last_time = unique_times.iloc[
    validation_end - 1
]

train_data = data[
    data["time_utc"] <= train_last_time
].copy()

validation_data = data[
    (data["time_utc"] > train_last_time)
    &
    (data["time_utc"] <= validation_last_time)
].copy()

test_data = data[
    data["time_utc"] > validation_last_time
].copy()

print("Training rows:", len(train_data))
print("Validation rows:", len(validation_data))
print("Test rows:", len(test_data))

Training rows: 159313
Validation rows: 34383
Test rows: 33522


Define Features and Target

In [5]:
feature_columns = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "cloud_cover",
    "wind_speed_10m",
    "global_tilted_irradiance",
    "diffuse_radiation",
    "sunshine_duration",
    "latitude",
    "longitude",
    "elevation",
    "hour_sin",
    "hour_cos",
    "day_of_year_sin",
    "day_of_year_cos",
    "gti_lag_1",
    "cloud_cover_lag_1"
]

target_column = "P"

Combine Training and Validation Data

In [6]:
final_train_data = pd.concat(
    [
        train_data,
        validation_data
    ],
    ignore_index=True
)

X_final_train = final_train_data[
    feature_columns
]

y_final_train = final_train_data[
    target_column
]

X_test = test_data[
    feature_columns
]

y_test = test_data[
    target_column
]

print(
    "Final training rows:",
    len(final_train_data)
)

print(
    "Test rows:",
    len(test_data)
)

print(
    "X_final_train:",
    X_final_train.shape
)

print(
    "X_test:",
    X_test.shape
)

Final training rows: 193696
Test rows: 33522
X_final_train: (193696, 17)
X_test: (33522, 17)


 Train Final Random Forest Model

In [7]:
final_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=2,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

final_model.fit(
    X_final_train,
    y_final_train
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",16
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.8
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples 

Final Test Evaluation

In [8]:
test_predictions = final_model.predict(
    X_test
)

test_mae = mean_absolute_error(
    y_test,
    test_predictions
)

test_mse = mean_squared_error(
    y_test,
    test_predictions
)

test_rmse = np.sqrt(
    test_mse
)

test_r2 = r2_score(
    y_test,
    test_predictions
)

print("Final Random Forest - Test Performance")
print("MAE:", round(test_mae, 3))
print("MSE:", round(test_mse, 3))
print("RMSE:", round(test_rmse, 3))
print("R²:", round(test_r2, 4))

Final Random Forest - Test Performance
MAE: 58.309
MSE: 7880.729
RMSE: 88.773
R²: 0.8197


Final Prediction Check

In [9]:
print(
    "Negative predictions:",
    (test_predictions < 0).sum()
)

print(
    "Minimum prediction:",
    test_predictions.min()
)

print(
    "Maximum prediction:",
    test_predictions.max()
)

Negative predictions: 0
Minimum prediction: 0.0006955328606657922
Maximum prediction: 705.491357814587


Test Performance Diagnostics

In [10]:
split_summary = pd.DataFrame({
    "Dataset": [
        "Training",
        "Validation",
        "Test"
    ],
    "Rows": [
        len(train_data),
        len(validation_data),
        len(test_data)
    ],
    "Mean P": [
        train_data["P"].mean(),
        validation_data["P"].mean(),
        test_data["P"].mean()
    ],
    "Std P": [
        train_data["P"].std(),
        validation_data["P"].std(),
        test_data["P"].std()
    ],
    "Mean GTI": [
        train_data["global_tilted_irradiance"].mean(),
        validation_data["global_tilted_irradiance"].mean(),
        test_data["global_tilted_irradiance"].mean()
    ]
})

split_summary.round(3)

,Dataset,Rows,Mean P,Std P,Mean GTI
0,Training,159313,318.340,225.864,458.117
1,Validation,34383,306.237,219.887,422.488
2,Test,33522,263.802,209.065,417.242


 Monthly Test Performance

In [11]:
test_analysis = test_data.copy()

test_analysis["prediction"] = test_predictions

test_analysis["month"] = (
    test_analysis["time_sl"]
    .dt.strftime("%Y-%m")
)

monthly_results = []

for month in sorted(
    test_analysis["month"].unique()
):
    month_data = test_analysis[
        test_analysis["month"] == month
    ]

    mae = mean_absolute_error(
        month_data["P"],
        month_data["prediction"]
    )

    mse = mean_squared_error(
        month_data["P"],
        month_data["prediction"]
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        month_data["P"],
        month_data["prediction"]
    )

    monthly_results.append({
        "Month": month,
        "Rows": len(month_data),
        "Mean Actual P": month_data["P"].mean(),
        "Mean Predicted P": month_data["prediction"].mean(),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

monthly_results_df = pd.DataFrame(
    monthly_results
)

monthly_results_df.round(3)

,Month,Rows,Mean Actual P,Mean Predicted P,MAE,MSE,RMSE,R2
0,2023-09,4516,292.542,311.220,51.748,6742.323,82.112,0.862
1,2023-10,9966,276.518,284.213,59.195,8161.118,90.339,0.828
2,2023-11,9345,247.895,267.119,62.057,8888.959,94.281,0.776
3,2023-12,9695,252.675,268.869,56.839,7150.947,84.563,0.823


District Test Performance

In [12]:
district_results = []

for district in sorted(
    test_analysis["district"].unique()
):
    district_data = test_analysis[
        test_analysis["district"] == district
    ]

    mae = mean_absolute_error(
        district_data["P"],
        district_data["prediction"]
    )

    mse = mean_squared_error(
        district_data["P"],
        district_data["prediction"]
    )

    rmse = np.sqrt(mse)

    r2 = r2_score(
        district_data["P"],
        district_data["prediction"]
    )

    district_results.append({
        "District": district,
        "Rows": len(district_data),
        "MAE": mae,
        "MSE": mse,
        "RMSE": rmse,
        "R2": r2
    })

district_results_df = pd.DataFrame(
    district_results
)

district_results_df.sort_values(
    "RMSE"
).round(3)

,District,Rows,MAE,MSE,RMSE,R2
20,Polonnaruwa,1329,50.469,5490.684,74.099,0.874
1,Anuradhapura,1345,53.357,6078.923,77.967,0.843
23,Trincomalee,1317,52.962,6357.419,79.733,0.864
3,Batticaloa,1323,55.777,6484.140,80.524,0.856
18,Mullaitivu,1321,52.982,6515.033,80.716,0.864
0,Ampara,1320,56.126,6721.334,81.984,0.844
12,Kilinochchi,1327,51.862,6803.567,82.484,0.837
24,Vavuniya,1332,55.642,7341.568,85.683,0.808
14,Mannar,1346,56.410,7468.118,86.418,0.826
2,Badulla,1332,57.419,7509.460,86.657,0.823


In [13]:
district_results_df.sort_values(
    "RMSE",
    ascending=False
).head(10).round(3)

,District,Rows,MAE,MSE,RMSE,R2
5,Galle,1361,65.052,10561.219,102.768,0.785
16,Matara,1365,62.104,10113.985,100.568,0.789
9,Kalutara,1354,65.295,9877.274,99.384,0.777
7,Hambantota,1360,60.477,9843.917,99.217,0.795
4,Colombo,1350,64.046,9178.885,95.806,0.791
8,Jaffna,1330,62.282,9159.975,95.708,0.784
10,Kandy,1346,61.727,8308.874,91.153,0.807
19,Nuwara Eliya,1343,60.567,8302.384,91.117,0.812
15,Matale,1350,61.432,8263.449,90.904,0.808
17,Monaragala,1330,58.503,8098.722,89.993,0.808


# Final Model

In [14]:
import os

os.makedirs(
    "../Models",
    exist_ok=True
)

joblib.dump(
    final_model,
    "../Models/final_random_forest_model.joblib"
)

print("Final model saved.")

Final model saved.


In [15]:
joblib.dump(
    feature_columns,
    "../Models/feature_columns.joblib"
)

print("Feature list saved.")

Feature list saved.


Verifying Saved Model

In [16]:
loaded_model = joblib.load(
    "../Models/final_random_forest_model.joblib"
)

loaded_features = joblib.load(
    "../Models/feature_columns.joblib"
)

loaded_predictions = loaded_model.predict(
    X_test[loaded_features]
)

print(
    "Predictions match:",
    np.allclose(
        test_predictions,
        loaded_predictions
    )
)

Predictions match: True
